## Installations and Settings 

In [1]:
# !pip install jedi
# !pip install -q llama-index-core
# !pip install -q llama-index-llms-groq
# !pip install -q llama-index-readers-file
# !pip install -q llama-index-embeddings-huggingface
# !pip install -q llama-index-embeddings-instructor
# !pip install python-dotenv

In [2]:
#setting up libraries and the Groq LLM
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

#model = "llama-3.1-8b-instant"
model = "openai/gpt-oss-120b"
#model = "openai/gpt-oss-20b"
#model = "llama-3.3-70b-versatile"

llm = Groq(
    model=model,
     api_key = os.environ.get(
       "GROQ_API_KEY"
     )
)

#loading documents
BASE_DIR = Path().resolve()

data_dirs = [
    BASE_DIR.parent.parent / "smyth-markdown" / "smyth-markdown-all.txt",
    BASE_DIR.parent.parent / "CrosbySchaeffer" / "chaps"
]

def load_all_documents(data_dirs):
    documents = []
    
    for d in data_dirs:
        print(f"Loading from: {d}")
        
        if not d.exists():
            raise ValueError(f"Path does not exist: {d}")
        
        if d.is_file():
            documents.extend(
                SimpleDirectoryReader(input_files=[str(d)]).load_data()
            )
        else:
            documents.extend(
                SimpleDirectoryReader(str(d)).load_data()
            )
    
    return documents

documents = load_all_documents(data_dirs)


#splitting documents into chunks

text_splitter = SentenceSplitter(chunk_size=800, chunk_overlap=150)

docs = text_splitter.get_nodes_from_documents(documents)


# embeddings
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings_folder = os.path.join(".", "embedding_model")

embeddings = HuggingFaceEmbedding(
    model_name=embedding_model, cache_folder=embeddings_folder
)


Loading from: C:\Users\Farnoosh\Documents\GitHub\smyth-markdown\smyth-markdown-all.txt
Loading from: C:\Users\Farnoosh\Documents\GitHub\CrosbySchaeffer\chaps


2026-04-15 18:26:22,994 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-04-15 18:26:25,079 - INFO - 2 prompts are loaded, with the keys: ['query', 'text']


### &nbsp; Creating/loading a vector database


In [3]:
# #creating the vector database
# vector_index = VectorStoreIndex.from_documents(
#     documents,
#     transformations=[text_splitter],
#     embed_model=embeddings
# )


# #saving the vector database
# vector_index.storage_context.persist(
#     persist_dir="vector-database"
# )

In [4]:
#loading the vector database
#from llama_index.storage.storage_context import StorageContext

from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Set the embedding model globally before loading
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

storage_context = StorageContext.from_defaults(persist_dir="vector-database")
vector_index = load_index_from_storage(storage_context)

2026-04-15 18:26:25,100 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-04-15 18:26:26,915 - INFO - 2 prompts are loaded, with the keys: ['query', 'text']
2026-04-15 18:26:33,531 - INFO - Loading all indices.


In [5]:
#retriever
retriever = vector_index.as_retriever(similarity_top_k=5)


### &nbsp; Adding a prompt


In [ ]:
#prompt template
input_template = """Here is the context:
{context_str}

you are an expert instructor of Ancient Greek. You are generating a structured textbook lesson based on a syllabus entry. you will be given a Morphological Type for explaining. Write a complete, self-contained lesson explaining this topic for students.
The lesson must include:
1. Clear explanation of the grammatical concept
2. Forms and patterns. add as many tables as you have available to illustrate the forms and patterns of the topic.
3. Key rules and exceptions
5. Example sentences with translations
6. Notes on usage and context
7. A short extra section on historical development and etymology (if applicable) 
Guidelines:
- Assume the student is a beginner to intermediate learner
- Be clear, structured, and didactic
- Use headings and create a properly formatted lesson
- Do NOT skip explanations even if the topic seems basic
- no excercise is needed, just the lesson content with one or two examples.

Output format: Markdown

 {query_str}
Answer:"""

prompt = PromptTemplate(template=input_template)

### &nbsp; RAG 

In [7]:
#rag
query_engine = vector_index.as_query_engine(
    llm=llm, text_qa_template=prompt, similarity_top_k=5, response_mode="tree_summarize"
)
#testing it out
#answer = query_engine.query("explain first declension with details and the morphology")
# print(answer)

### Looping over a sample syllabus to generate grammar modules

In [ ]:
# Simple batch generator: read CSV rows and generate lessons (max 20 per day).
import csv
import os
import re
import time
import json
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

CSV_PATH = os.path.join(".", "frequency_syllabus.csv")
OUTPUT_DIR = os.path.join("..", "lessons-no-decl")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, ".progress.json")
DAILY_QUOTA = 20
DELAY_BETWEEN_REQUESTS = 10
MAX_SECONDS_PER_ROW = 180

os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_filename(name):
    name = str(name).lower()
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.strip("_")

def get_topic(row):
    return (
        row.get("morph_type")
        or row.get("syllabus")
        or row.get("topic")
        or row.get("name")
        or "Unknown topic"
    )

def get_pos(row):
    return row.get("pos") or row.get("pos_category") or ""

def get_frequency(row):
    return row.get("token_count") or row.get("frequency") or row.get("pct") or ""

def lesson_key(row):
    # Use topic slug so existing legacy files (e.g., accusative.md) are reused.
    topic_slug = safe_filename(get_topic(row)) or "unknown_topic"
    return topic_slug

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        try:
            with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
            return {
                "completed": set(data.get("completed", [])),
                "failed": data.get("failed", {}),
            }
        except Exception:
            pass
    return {"completed": set(), "failed": {}}

def save_progress(progress):
    data = {
        "completed": sorted(progress["completed"]),
        "failed": progress["failed"],
        "updated_at": datetime.now().isoformat(timespec="seconds"),
    }
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def build_prompt(row):
    topic = get_topic(row)

    return f"""You are a Greek language tutor writing a detailed, pedagogically clear lesson for a student textbook.

Write a complete, self-contained lesson explaining **{topic}** for students.
The lesson must include:
1. Clear explanation of the grammatical concept
2. Forms and patterns. add as many tables as you have available to illustrate the forms and patterns of the topic.
3. Key rules and exceptions. again, having the paradigm (full declension table or conjugation table as appropriate) is very important here.
5. Example sentences with translations
6. Notes on usage and context
7. A short extra section on historical development and etymology (if applicable) 
Guidelines:
- Assume the student is a beginner to intermediate learner
- Be clear, structured, and didactic
- Use headings and create a properly formatted lesson
- Do NOT skip explanations even if the topic seems basic.
- no excercise is needed, just the lesson content with one or two examples.

Output format:
Write in Markdown format with clear headings. Assume the student has familiarity with the Greek alphabet.
"""

def generate_lesson(prompt, query_engine, timeout_seconds):
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(query_engine.query, prompt)
        try:
            answer = future.result(timeout=timeout_seconds)
        except FuturesTimeoutError as exc:
            raise TimeoutError(f"Row timed out after {timeout_seconds} seconds") from exc

    return answer.response if hasattr(answer, "response") else str(answer)

def count_today_outputs(output_dir):
    today = datetime.now().date()
    count = 0
    for name in os.listdir(output_dir):
        if not name.endswith(".md"):
            continue
        path = os.path.join(output_dir, name)
        if os.path.isfile(path):
            mtime_date = datetime.fromtimestamp(os.path.getmtime(path)).date()
            if mtime_date == today:
                count += 1
    return count

def existing_lesson_keys(output_dir):
    keys = set()
    for name in os.listdir(output_dir):
        if name.endswith(".md"):
            keys.add(os.path.splitext(name)[0])
    return keys

def run_pipeline(query_engine):
    with open(CSV_PATH, newline="", encoding="utf-8") as csvfile:
        reader = csv.DictReader(csvfile)
        rows = list(reader)

    if not rows:
        return

    progress = load_progress()
    existing_keys = existing_lesson_keys(OUTPUT_DIR)
    progress["completed"] = progress["completed"].intersection(existing_keys)
    save_progress(progress)

    generated_today = count_today_outputs(OUTPUT_DIR)
    remaining_quota = max(0, DAILY_QUOTA - generated_today)

    if remaining_quota <= 0:
        return

    pending = []
    for idx, row in enumerate(rows, start=1):
        key = lesson_key(row)
        filename = key + ".md"
        filepath = os.path.join(OUTPUT_DIR, filename)

        if os.path.exists(filepath):
            progress["completed"].add(key)
            continue
        if key in progress["completed"]:
            progress["completed"].discard(key)

        pending.append((idx, row))

    save_progress(progress)

    if not pending:
        return

    to_process = pending[:remaining_quota]
    generated_files = []

    for i, (idx, row) in enumerate(to_process, 1):
        topic = get_topic(row)
        key = lesson_key(row)
        filename = key + ".md"
        filepath = os.path.join(OUTPUT_DIR, filename)

        try:
            prompt = build_prompt(row)
            lesson_text = generate_lesson(prompt, query_engine, MAX_SECONDS_PER_ROW)

            with open(filepath, "w", encoding="utf-8") as f:
                f.write("---\n")
                f.write(f"topic: {topic}\n")
                f.write(f"pos: {get_pos(row)}\n")
                f.write(f"frequency: {get_frequency(row)}\n")
                f.write("---\n\n")
                f.write(lesson_text)

            progress["completed"].add(key)
            progress["failed"].pop(key, None)
            save_progress(progress)
            generated_files.append(filename)
            print(filename)
        except Exception as e:
            progress["failed"][key] = str(e)
            save_progress(progress)

        if i < len(to_process):
            time.sleep(DELAY_BETWEEN_REQUESTS)

    print(f"Generated files: {len(generated_files)}")

query_engine = vector_index.as_query_engine(
    llm=llm,
    text_qa_template=prompt,
    similarity_top_k=3,
    response_mode="compact"
)

run_pipeline(query_engine)

2026-04-15 18:47:58,723 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


aorist_participle_active_irregulars.md


2026-04-15 18:48:17,430 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


aorist_participle_active_deponent.md


2026-04-15 18:48:35,771 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


...

perfect_participle_active_irregulars.md


2026-04-15 18:51:51,808 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-04-15 18:51:51,812 - INFO - Retrying request to /chat/completions in 8.000000 seconds
2026-04-15 18:52:07,197 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


perfect_indicative_middlepassive_ω.md


2026-04-15 18:52:17,618 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-04-15 18:52:17,622 - INFO - Retrying request to /chat/completions in 25.000000 seconds
2026-04-15 18:52:50,489 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


future_infinitive_active_ω.md
Generated files: 10
